In [1]:
import os

In [2]:
%pwd

'c:\\Users\\lokar\\Desktop\\End-to-End-Machine-Learning-Project-with-ML-FLOW\\research'

In [3]:
os.chdir("../")

In [4]:
%pwd

'c:\\Users\\lokar\\Desktop\\End-to-End-Machine-Learning-Project-with-ML-FLOW'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class ModelTrainerConfig:
    root_dir: Path
    train_data_path: Path
    test_data_path: Path
    model_name: str
    alpha: float
    l1_ratio: float
    target_column: str

In [6]:
from mlproject.constants import *
from mlproject.utils.common import read_yaml, create_directories

In [7]:
class ConfigurationManager:
    def __init__(
            self,
            config_filepath: Path = CONFIG_FILE_PATH,
            params_filepath: Path = PARAMS_FILE_PATH,
            schema_filepath: Path = SCHEMA_FILE_PATH):

            self.config = read_yaml(config_filepath)
            self.params = read_yaml(params_filepath)
            self.schema = read_yaml(schema_filepath)

            create_directories([self.config.artifacts_root])


    def get_model_trainer_config(self) -> ModelTrainerConfig:
          config = self.config.model_trainer
          params = self.params.ElasticNet
          schema = self.schema.TARGET_COLUMN

          create_directories([config.root_dir])

          model_trainer_config = ModelTrainerConfig(
                root_dir=config.root_dir,
                train_data_path=config.train_data_path,
                test_data_path=config.test_data_path,
                model_name=config.model_name,
                alpha=params.alpha,
                l1_ratio=params.l1_ratio,
                target_column=schema.name
          )

          return model_trainer_config

In [8]:
import pandas as pd
import os
from mlproject import logger
from sklearn.linear_model import ElasticNet
import joblib

In [ ]:
class ModelTrainer:
    def __init__(self, config: ModelTrainerConfig):
        self.config = config

    def train_model(self):
        logger.info("Reading training data")
        train_df = pd.read_csv(self.config.train_data_path)
        logger.info("Reading testing data")
        test_df = pd.read_csv(self.config.test_data_path)

        logger.info("Splitting features and target variable")
        X_train = train_df.drop(columns=[self.config.target_column], axis=1)
        y_train = train_df[self.config.target_column]

        X_test = test_df.drop(columns=[self.config.target_column], axis=1)
        y_test = test_df[self.config.target_column]

        logger.info("Training the model")
        model = ElasticNet(alpha=self.config.alpha, l1_ratio=self.config.l1_ratio)
        model.fit(X_train, y_train)

        logger.info("Saving the trained model")
        model_path = os.path.join(self.config.root_dir, self.config.model_name)
        joblib.dump(model, model_path)

        logger.info(f"Model saved at {model_path}")

In [12]:
try:
    config = ConfigurationManager()
    model_trainer_config = config.get_model_trainer_config()
    model_trainer_config = ModelTrainer(config=model_trainer_config)
    model_trainer_config.train_model()
except Exception as e:
    raise e

[2026-10-27 16:26:23,086: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-10-27 16:26:23,098: INFO: common: yaml file: params.yaml loaded successfully]
[2026-10-27 16:26:23,103: INFO: common: yaml file: schema.yaml loaded successfully]
[2026-10-27 16:26:23,107: INFO: common: created directory at: artifacts]
[2026-10-27 16:26:23,117: INFO: common: created directory at: artifacts/model_trainer]
[2026-10-27 16:26:23,123: INFO: 1577123017: Reading training data]
[2026-10-27 16:26:23,468: INFO: 1577123017: Reading testing data]
[2026-10-27 16:26:23,499: INFO: 1577123017: Splitting features and target variable]
[2026-10-27 16:26:23,585: INFO: 1577123017: Training the model]
[2026-10-27 16:26:24,838: INFO: 1577123017: Saving the trained model]
[2026-10-27 16:26:24,858: INFO: 1577123017: Model saved at artifacts/model_trainer\model.joblib]
